# SmogCast – обучение LSTM (Google Colab, GPU)

Нейросеть LSTM прогнозирует почасовой PM2.5 в Алматы на 72 часа по последней неделе ряда и календарю;
второй вариант дополнительно получает прогноз погоды (температура, ветер, осадки).

**Режимы обучения** (по умолчанию за один запуск – `test` и `live`):
- `validation` – обучение до 01.10.2024, для подбора и проверки на валидации (уже выполнен, вывод –
  в `notebooks/lstm_validation.ipynb`);
- `test` – обучение до 01.10.2025, для итоговой проверки на отложенном тестовом годе;
- `live` – обучение на всех данных снимка, для прогнозов на сайте.

**Как запустить**
1. «Среда выполнения» → «Сменить среду выполнения» → **T4 GPU** → «Сохранить».
2. «Среда выполнения» → **«Выполнить все»** (несколько минут).
3. В конце браузер скачает `lstm_artifacts.zip` с папками режимов – распакуйте его в
   `SmogCast/model/artifacts/` (получится `artifacts/test/…` и `artifacts/live/…`).

Ключи не нужны: код и данные – из публичного репозитория [AlexTrav/SmogCast](https://github.com/AlexTrav/SmogCast),
те же `prep.py`, `weather.py` и `lstm.py`, что у ARIMA и Prophet.

**Честность проверки.** Модель видит только данные до конца обучения своего режима; ранняя остановка – по двум
отложенным блокам внутри этих данных (зима и конец лета последнего года). Тестовый год здесь не оценивается –
итоговая проверка всех моделей идёт один раз, локально. При обучении будущая погода – фактическая
(«идеальный прогноз»), при проверке – только прогнозы, опубликованные к моменту прогноза.

In [ ]:
# код и данные проекта – из публичного репозитория
!rm -rf /content/SmogCast && git clone -q --depth 1 https://github.com/AlexTrav/SmogCast.git /content/SmogCast
!pip -q install onnx==1.23.2 onnxruntime==1.31.1

import os
import sys

os.environ["SERIES"] = "/content/SmogCast/data/series/almaty.csv"
os.environ["WEATHER_PREV"] = "/content/SmogCast/data/series/weather_prev.csv"
sys.path[:0] = ["/content/SmogCast/research", "/content/SmogCast/model"]

import torch

print("PyTorch", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "нет – включите T4 GPU в «Среда выполнения → Сменить среду выполнения»")

In [ ]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd

import backtest
import lstm
import prep

frame = prep.clean(prep.load())  # снимок ряда, местное время, сбой CAMS заменён интерполяцией

# режим -> конец обучающих данных (цели всех примеров строго раньше этой даты)
MODES = {
    "test": backtest.PERIODS["test"][0],  # 01.10.2025 – начало отложенного теста
    "live": frame.index[-1] + pd.Timedelta(hours=1),  # все данные снимка
}
# MODES = {"validation": backtest.VALID_START}  # повтор первого запуска


# отложенные блоки для ранней остановки внутри обучающих данных: зима и конец лета последнего года;
# примеры, чьё окно (неделя истории + 72 ч цели) задевает блок, в обучение не берутся
def split(train_end):
    year = (train_end - pd.Timedelta(days=60)).year
    blocks = [(pd.Timestamp(f"{year}-01-08", tz=prep.TZ), pd.Timestamp(f"{year}-02-19", tz=prep.TZ)),
              (pd.Timestamp(f"{year}-08-01", tz=prep.TZ), train_end)]
    positions = lstm.train_positions(frame.index, train_end)
    times = frame.index[positions]
    margin_past, margin_future = pd.Timedelta(hours=lstm.LOOKBACK), pd.Timedelta(hours=lstm.HORIZON)
    hold = np.zeros(len(times), bool)
    touch = np.zeros(len(times), bool)
    for a, b in blocks:
        hold |= (times >= a) & (times < b)
        touch |= (times >= a - margin_future) & (times < b + margin_past)
    return positions[~touch], positions[hold]


for mode, train_end in MODES.items():
    fit_positions, hold_positions = split(train_end)
    print(f"{mode}: данные {frame.index[0]:%d.%m.%Y} – {train_end:%d.%m.%Y %H:%M}, "
          f"обучающих примеров {len(fit_positions)}, для ранней остановки {len(hold_positions)}")

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"


class Net(torch.nn.Module):
    """LSTM читает неделю истории; её итоговое состояние вместе с признаками будущих 72 ч
    (календарь, прогноз погоды) превращает полносвязная голова в 72 приращения ln PM2.5."""

    def __init__(self, past_features: int, future_features: int, hidden: int = 64) -> None:
        super().__init__()
        self.lstm = torch.nn.LSTM(past_features, hidden, num_layers=2, batch_first=True, dropout=0.1)
        self.head = torch.nn.Sequential(
            torch.nn.Linear(hidden + lstm.HORIZON * future_features, 256), torch.nn.ReLU(),
            torch.nn.Dropout(0.1), torch.nn.Linear(256, lstm.HORIZON))

    def forward(self, past, future):
        _, (state, _) = self.lstm(past)
        return self.head(torch.cat([state[-1], future.flatten(1)], dim=1))


def tensors(arrays):
    return [torch.from_numpy(a).to(device) for a in arrays]


def train_variant(variables, train_end, epochs=60, patience=6, batch=256):
    torch.manual_seed(0)
    np.random.seed(0)
    fit_positions, hold_positions = split(train_end)
    norm = lstm.fit_norm(frame[frame.index < train_end], variables)
    fit = tensors(lstm.build_samples(frame, fit_positions, variables, norm))
    hold = tensors(lstm.build_samples(frame, hold_positions, variables, norm))
    model = Net(fit[0].shape[2], fit[1].shape[2]).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
    loss_fn = torch.nn.L1Loss()  # средняя абсолютная ошибка в логарифмах ~ относительная ошибка
    best, best_state, waited, log = float("inf"), None, 0, []
    for epoch in range(1, epochs + 1):
        began = time.time()
        model.train()
        order = torch.randperm(len(fit[0]), device=device)
        for i in range(0, len(order), batch):
            idx = order[i:i + batch]
            optimizer.zero_grad()
            loss = loss_fn(model(fit[0][idx], fit[1][idx]), fit[2][idx])
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        model.eval()
        with torch.no_grad():
            hold_loss = float(loss_fn(model(hold[0], hold[1]), hold[2]))
        log.append(hold_loss)
        print(f"  эпоха {epoch:2d}: ошибка на отложенных блоках {hold_loss:.4f} ({time.time() - began:.1f} с)")
        if hold_loss < best - 1e-4:
            best, waited = hold_loss, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            waited += 1
            if waited >= patience:
                print("  ранняя остановка")
                break
    model.load_state_dict(best_state)
    return model.eval(), norm, {"hold_loss": round(best, 5), "epochs": epoch, "hold_curve": [round(x, 5) for x in log]}

In [ ]:
import onnxruntime as ort

VARIANTS = {
    "lstm": ((), "LSTM"),
    "lstm_weather": (lstm.WEATHER, "LSTM + прогноз погоды (температура, ветер, осадки)"),
}
summary = []
for mode, train_end in MODES.items():
    Path(f"artifacts/{mode}").mkdir(parents=True, exist_ok=True)
    for key, (variables, name) in VARIANTS.items():
        print(f"\n{mode}: {name}")
        model, norm, info = train_variant(variables, train_end)
        model = model.cpu()
        past = torch.randn(4, lstm.LOOKBACK, 5 + len(variables))
        future = torch.randn(4, lstm.HORIZON, 4 + len(variables))
        path = f"artifacts/{mode}/{key}.onnx"
        kwargs = dict(input_names=["past", "future"], output_names=["delta"], opset_version=17,
                      dynamic_axes={"past": {0: "batch"}, "future": {0: "batch"}, "delta": {0: "batch"}})
        try:
            torch.onnx.export(model, (past, future), path, dynamo=False, **kwargs)
        except TypeError:  # в старых версиях PyTorch нет параметра dynamo
            torch.onnx.export(model, (past, future), path, **kwargs)
        # ONNX должен считать то же, что PyTorch
        session = ort.InferenceSession(path, providers=["CPUExecutionProvider"])
        onnx_out = session.run(None, {"past": past.numpy(), "future": future.numpy()})[0]
        with torch.no_grad():
            diff = float(np.abs(onnx_out - model(past, future).numpy()).max())
        assert diff < 1e-4, f"ONNX расходится с PyTorch: {diff}"
        meta = {"name": name, "mode": mode, "weather": list(variables), "norm": norm, "lookback": lstm.LOOKBACK,
                "horizon": lstm.HORIZON, "trained_until": str(train_end), "torch": torch.__version__,
                "onnx_max_diff": diff, "created": pd.Timestamp.now(tz="UTC").isoformat(), **info}
        Path(f"artifacts/{mode}/{key}.json").write_text(json.dumps(meta, ensure_ascii=False, indent=1),
                                                        encoding="utf-8")
        summary.append({"режим": mode, "модель": name, "эпох": info["epochs"], "ошибка остановки": info["hold_loss"]})
        print(f"  сохранено: {path} ({Path(path).stat().st_size / 1024:.0f} КБ), расхождение с PyTorch {diff:.1e}")

pd.set_option("display.width", 200)
print()
print(pd.DataFrame(summary).to_string(index=False))

In [ ]:
!rm -f lstm_artifacts.zip && cd artifacts && zip -q -r ../lstm_artifacts.zip . && cd .. && unzip -l lstm_artifacts.zip
from google.colab import files

files.download("lstm_artifacts.zip")